# Knowledge Distillation (RKD v2): ConvNeXt V2 → MobileNetV3 (Colab)

**Cải tiến so với v1:** Thêm `ProjectionHead` để tách biệt không gian embedding của student và teacher.

| | Teacher | Student |
|---|---|---|
| Model | `MTLFaceRecognition` (ConvNeXt V2) | `FaceRecognitionMobileNetV3` |
| Params | ~28M | ~5M |
| Embedding | `x_id` (512-D) | 512-D |
| Mode | **Frozen** | **Trainable** |

**Luồng forward:**
```
student_emb  ──→  MagLinear  ──→  L_MagFace      (task loss — space riêng của student)
student_emb  ──→  projector  ──→  proj_emb
                                   ↕↕↕
                              teacher_emb          (KD + RKD-D + RKD-A)
```

**Loss tổng hợp:**
```
L_total = α·L_MagFace  +  β·L_KD_cosine  +  γ·L_RKD_D  +  δ·L_RKD_A

L_MagFace    : task loss trên student_emb (space riêng)
L_KD_cosine  : 1 - cosine_sim(proj_emb, teacher_emb)  — point-wise
L_RKD_D      : Huber(dist_s(i,j)/μ_s − dist_t(i,j)/μ_t)  — pairwise trên proj_emb
L_RKD_A      : Huber(cos∠_s(i,j,k) − cos∠_t(i,j,k))     — triplet trên proj_emb
```

**Tại sao ProjectionHead giúp ích:**
- Không có projector: `student_emb` bị kéo đồng thời theo MagFace và KD → hai mục tiêu mâu thuẫn
- Có projector: `student_emb` tự do tối ưu cho MagFace, `projector` học ánh xạ sang space của teacher
- RKD trên `proj_emb` bảo toàn cấu trúc hình học trong space đã bridge

**Cách dùng:**
1. Mount Google Drive
2. Sửa `CONFIGURATION` và `TEACHER_CKPT`
3. Chạy từ trên xuống

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

TEACHER_CKPT = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

EXPERIMENT_NAME = 'KD_RKD_v2_ConvNextV2_to_MobileNetV3_Albedo'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,
    'output_dir':  '/content/drive/MyDrive/',

    # Modality: 'albedo' | 'normalmap' | 'depthmap'
    'type':        'albedo',

    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',

    'use_sampler': True,
    'device':      device,
    'epochs':      40,
    'batch_size':  32,   # RKD-A tạo tensor O(B³) — không tăng quá 64
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,

    # Trọng số loss (reduction='mean' cho rkd_d và rkd_a)
    #
    # Raw value khi converged (epoch40), proj_emb sau projector:
    #   loss_task    ≈ 3.2   → × task_weight = 6.4  (~75%)
    #   loss_kd      ≈ 0.05  → × kd_weight   = 0.05 (proj_emb gần teacher_emb hơn)
    #   loss_rkd_d   ≈ 0.002 → × rkd_d_weight
    #   loss_rkd_a   ≈ 0.002 → × rkd_a_weight
    #
    # Bảng tham khảo rkd weights:
    #   weight =  25 → contribution ≈ 0.05  (~0.6%)  ← nhẹ
    #   weight = 160 → contribution ≈ 0.32  (~4.0%)  ← trung bình
    #   weight = 320 → contribution ≈ 0.64  (~8.0%)  ← mạnh
    #   (paper gốc: rkd_a_weight = 2 × rkd_d_weight)
    'task_weight':  2.0,
    'kd_weight':    1.0,
    'rkd_d_weight': 25.0,
    'rkd_a_weight': 50.0,
}

print(f"Dataset dir : {CONFIGURATION['dataset_dir']}")
print(f"Output dir  : {CONFIGURATION['output_dir']}")
print(f"Teacher ckpt: {TEACHER_CKPT}")
print(f"RKD weights : D={CONFIGURATION['rkd_d_weight']}  A={CONFIGURATION['rkd_a_weight']}  (mean reduction)")

## 3. Data Loading

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(
        f'Không tìm thấy CSV train tại {dataset_dir}.\n'
        f'Kiểm tra lại DRIVE_DATASET_DIR.'
    )
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, test_transform)

## 4. Teacher Model (ConvNeXt V2 — Frozen)

In [ ]:
if not os.path.exists(TEACHER_CKPT):
    raise FileNotFoundError(
        f'Không tìm thấy teacher checkpoint: {TEACHER_CKPT}\n'
        f'Kiểm tra lại biến TEACHER_CKPT.'
    )

teacher = MTLFaceRecognition(
    backbone=CONFIGURATION['teacher_backbone'],
    num_classes=CONFIGURATION['num_classes'],
)

ckpt = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
state_dict = ckpt['model_state_dict']
keys_to_remove = [k for k in state_dict.keys() if 'id_head.maglinear' in k]
for k in keys_to_remove:
    del state_dict[k]
teacher.load_state_dict(state_dict, strict=False)
print(f"Teacher loaded — epoch {ckpt.get('epoch', '?')}")

teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'Teacher params: {teacher_params:,} (tất cả frozen)')

with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    _t_emb = teacher.get_embedding(_dummy)[-1]
    print(f'Teacher ID embedding shape: {_t_emb.shape}')

## 4.1. Đánh giá Teacher Model (baseline)

In [ ]:
class _TeacherEvalWrapper(torch.nn.Module):
    def __init__(self, teacher):
        super().__init__()
        self._teacher = teacher

    def get_embedding(self, x):
        return self._teacher.get_embedding(x)[-1]


teacher_wrapper = _TeacherEvalWrapper(teacher).to(device)
teacher_wrapper.eval()

teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)
teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{teacher_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{teacher_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{teacher_gp_rank1:.4f}"],
]
print(f"Teacher ({CONFIGURATION['teacher_backbone']}) — {TEACHER_CKPT}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

## 5. Student Model (MobileNetV3 — Trainable)

In [ ]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

with torch.no_grad():
    _s_emb = student.get_embedding(_dummy)
    print(f'Student embedding shape: {_s_emb.shape}')

## 6. Projection Head

MLP nhỏ bridge student embedding space → teacher embedding space.

```
student_emb (512-D)  →  projector  →  proj_emb (512-D)  ↔  teacher_emb
```

- `student_emb` vẫn đi thẳng vào `MagLinear` (task loss không thay đổi)
- `proj_emb` dùng cho **tất cả** KD losses: cosine + RKD-D + RKD-A
- Projector **chỉ tồn tại khi train**, bỏ hoàn toàn khi export ONNX

In [ ]:
class ProjectionHead(nn.Module):
    """
    Linear(512→512) → BN → ReLU → Linear(512→512)
    Bridge student embedding space sang teacher embedding space.
    """
    def __init__(self, in_dim: int = 512, hidden_dim: int = 512, out_dim: int = 512):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


projector = ProjectionHead(in_dim=512, hidden_dim=512, out_dim=512).to(device)
proj_params = sum(p.numel() for p in projector.parameters())
print(f'ProjectionHead params: {proj_params:,}  (train-only, dropped at ONNX export)')

with torch.no_grad():
    _proj_emb = projector(_s_emb)
    print(f'proj_emb shape: {_proj_emb.shape}')

## 7. RKD Loss

```
L_total = α·L_MagFace + β·L_KD_cosine + γ·L_RKD_D + δ·L_RKD_A
```

| Loss | Input | Ý nghĩa |
|---|---|---|
| `L_MagFace` | `student_emb` | Phân biệt class boundary |
| `L_KD_cosine` | `proj_emb` ↔ `teacher_emb` | Point-wise: hướng embedding |
| `L_RKD_D` | `proj_emb` ↔ `teacher_emb` | Pairwise: tỉ lệ khoảng cách |
| `L_RKD_A` | `proj_emb` ↔ `teacher_emb` | Triplet: góc tại đỉnh giữa |

**RKD-D** (Eq.7, Park et al. 2019):
$$\mathcal{L}_{RKD-D} = \frac{1}{|\mathcal{P}^2|} \sum_{i \neq j} l_\delta\!\left(\frac{d_s(i,j)}{\mu_s},\, \frac{d_t(i,j)}{\mu_t}\right)$$

**RKD-A** (Eq.10): góc tại đỉnh **j** (đỉnh giữa):
$$\mathcal{L}_{RKD-A} = \frac{1}{|\mathcal{P}^3|} \sum_{(i,j,k)} l_\delta\!\left(\cos\angle_s(i,j,k),\, \cos\angle_t(i,j,k)\right)$$
$$\cos\angle(i,j,k) = \langle e^{ij},\, e^{kj}\rangle, \quad e^{ij} = \frac{e_i - e_j}{\|e_i - e_j\|}$$

In [ ]:
class RKDLoss(nn.Module):
    """
    L_total = task_w*L_MagFace(student_emb) + kd_w*L_KD + rkd_d_w*L_RKD_D + rkd_a_w*L_RKD_A
    L_KD, L_RKD_D, L_RKD_A đều tính trên proj_emb ↔ teacher_emb.
    reduction='mean' — ổn định khi thay đổi batch size.
    """

    def __init__(
        self,
        metadata_path: str,
        task_weight:   float = 2.0,
        kd_weight:     float = 1.0,
        rkd_d_weight:  float = 25.0,
        rkd_a_weight:  float = 50.0,
    ):
        super().__init__()
        self.magface  = WeightClassMagLoss(metadata_path)
        self.task_w   = task_weight
        self.kd_w     = kd_weight
        self.rkd_d_w  = rkd_d_weight
        self.rkd_a_w  = rkd_a_weight

    # ── helpers ──────────────────────────────────────────────────────────

    @staticmethod
    def _pdist(e: torch.Tensor) -> torch.Tensor:
        """Pairwise L2 distance matrix [B, B]."""
        diff = e.unsqueeze(0) - e.unsqueeze(1)
        return diff.pow(2).sum(-1).clamp(min=1e-12).sqrt()

    def _rkd_distance(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """
        RKD-D: mean Huber(ψ_D(s_i,s_j), ψ_D(t_i,t_j)) trên các cặp i≠j.
        ψ_D = dist / μ,  μ = mean pairwise dist off-diagonal.
        """
        B = s_emb.size(0)
        mask = ~torch.eye(B, dtype=torch.bool, device=s_emb.device)

        with torch.no_grad():
            td   = self._pdist(t_emb)
            mu_t = td[mask].mean()
            td_n = td / (mu_t + 1e-8)

        sd   = self._pdist(s_emb)
        mu_s = sd[mask].mean()
        sd_n = sd / (mu_s + 1e-8)

        return F.huber_loss(sd_n[mask], td_n[mask], delta=1.0, reduction='mean')

    def _rkd_angle(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """
        RKD-A: mean Huber(ψ_A(s), ψ_A(t)) trên tất cả bộ ba.
        Góc tại đỉnh j (đỉnh giữa, Park et al. 2019 Eq.9):
          diff[j,i] = e[i]-e[j]  →  e^{ij} (unit vector từ j đến i)
          angle[j,i,k] = dot(e^{ij}, e^{kj})
        """
        def _angle_matrix(e):
            diff = e.unsqueeze(0) - e.unsqueeze(1)          # [B,B,D], diff[j,i]=e[i]-e[j]
            norm = diff.norm(p=2, dim=2, keepdim=True).clamp(min=1e-8)
            diff = diff / norm
            return torch.bmm(diff, diff.transpose(1, 2))    # [B,B,B]

        with torch.no_grad():
            ta = _angle_matrix(t_emb)
        sa = _angle_matrix(s_emb)
        return F.huber_loss(sa, ta, delta=1.0, reduction='mean')

    # ── forward ──────────────────────────────────────────────────────────

    def forward(
        self,
        student_logits,   # từ MagLinear(student_emb)
        student_norm,
        proj_emb,         # [B,512] — student_emb sau projector
        teacher_emb,      # [B,512] — frozen teacher
        id_labels,
    ):
        l_task = self.magface(student_logits, id_labels, student_norm)

        s_n  = F.normalize(proj_emb,   p=2, dim=1)
        t_n  = F.normalize(teacher_emb, p=2, dim=1)
        l_kd = (1.0 - F.cosine_similarity(s_n, t_n, dim=1)).mean()

        l_rkd_d = self._rkd_distance(proj_emb, teacher_emb)
        l_rkd_a = self._rkd_angle(proj_emb, teacher_emb)

        total = (
            self.task_w  * l_task
          + self.kd_w    * l_kd
          + self.rkd_d_w * l_rkd_d
          + self.rkd_a_w * l_rkd_a
        )
        return total, l_task, l_kd, l_rkd_d, l_rkd_a


criterion = RKDLoss(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
    rkd_d_weight=CONFIGURATION['rkd_d_weight'],
    rkd_a_weight=CONFIGURATION['rkd_a_weight'],
)
print('RKDLoss khởi tạo thành công.')
print(f"  task={CONFIGURATION['task_weight']}  kd={CONFIGURATION['kd_weight']}  "
      f"rkd_d={CONFIGURATION['rkd_d_weight']}  rkd_a={CONFIGURATION['rkd_a_weight']}")

## 8. Training

In [ ]:
def train_epoch(train_dl, teacher, student, projector, criterion, optimizer, device):
    student.train()
    projector.train()

    total_loss = total_task = total_kd = total_rkd_d = total_rkd_a = 0.0

    for X, y in train_dl:
        X, y      = X.to(device), y.to(device)
        id_labels = y[:, 0]

        with torch.no_grad():
            teacher_emb = teacher.get_embedding(X)[-1]   # [B, 512]

        feat        = student.backbone(X)
        student_emb = student.embedding(feat)            # [B, 512] — cho MagLinear
        proj_emb    = projector(student_emb)             # [B, 512] — bridge sang teacher space
        logits, norm = student.maglinear(student_emb)   # task loss dùng student_emb gốc

        loss, l_task, l_kd, l_rkd_d, l_rkd_a = criterion(
            logits, norm, proj_emb, teacher_emb, id_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        total_task  += l_task.item()
        total_kd    += l_kd.item()
        total_rkd_d += l_rkd_d.item()
        total_rkd_a += l_rkd_a.item()

    n = len(train_dl)
    return (
        total_loss  / n,
        total_task  / n,
        total_kd    / n,
        total_rkd_d / n,
        total_rkd_a / n,
    )


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
optimizer = Adam(
    list(student.parameters()) + list(projector.parameters()),
    lr=CONFIGURATION['base_lr'],
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Teacher: {CONFIGURATION['teacher_backbone']} | "
    f"Student: {CONFIGURATION['backbone']} | "
    f"Modality: {CONFIGURATION['type']} | "
    f"task={CONFIGURATION['task_weight']} kd={CONFIGURATION['kd_weight']} "
    f"rkd_d={CONFIGURATION['rkd_d_weight']} rkd_a={CONFIGURATION['rkd_a_weight']} | "
    f"projector=True"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=20,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')

In [ ]:
START_EPOCH = 0

manager.log_text('BAT DAU RKD v2 KNOWLEDGE DISTILLATION (ProjectionHead)')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    train_loss, train_task, train_kd, train_rkd_d, train_rkd_a = train_epoch(
        train_dl, teacher, student, projector, criterion, optimizer, device
    )

    # AUC đo trên student.get_embedding() — không qua projector
    train_auc = compute_id_auc(train_dl, student, device)
    test_auc  = compute_id_auc(test_dl,  student, device)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd':          train_kd,
        'loss_rkd_d':       train_rkd_d,
        'loss_rkd_a':       train_rkd_a,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    writer.add_scalar('Loss/total',   train_loss,   epoch + 1)
    writer.add_scalar('Loss/task',    train_task,   epoch + 1)
    writer.add_scalar('Loss/kd',      train_kd,     epoch + 1)
    writer.add_scalar('Loss/rkd_d',   train_rkd_d,  epoch + 1)
    writer.add_scalar('Loss/rkd_a',   train_rkd_a,  epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver(
        student, optimizer, epoch + 1, test_metrics, scheduler,
        extra_state={'projector_state_dict': projector.state_dict()},
    )
    early_stopping(test_metrics, student, epoch + 1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('RKD v2 KNOWLEDGE DISTILLATION HOAN TAT.')

## 9. Resume Training từ Checkpoint

> Chạy cell Setup → Imports → Data → Teacher → Student → Projector → Loss → Setup Train,  
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

checkpoint = torch.load(CKPT_PATH, map_location=device)
student.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
if 'projector_state_dict' in checkpoint:
    projector.load_state_dict(checkpoint['projector_state_dict'])
    print('Projector state loaded.')
else:
    print('Không tìm thấy projector_state_dict — projector khởi tạo ngẫu nhiên.')

START_EPOCH = checkpoint['epoch']
print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

## 10. Đánh giá Final

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{student_gp_rank1:.4f}"],
]
print(f"\nStudent ({CONFIGURATION['backbone']}) — RKD v2 from {CONFIGURATION['teacher_backbone']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

## 10.1. So sánh Teacher vs Student

In [ ]:
compare_rows = [
    ['Model',                          CONFIGURATION['teacher_backbone'],           CONFIGURATION['backbone']],
    ['Params',                         f'{teacher_params:,}',                       f'{total_p:,}'],
    ['Cosine AUC    (gallery→probe)',   f"{teacher_gp_auc['id_cosine']:.4f}",        f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)',   f"{teacher_gp_auc['id_euclidean']:.4f}",     f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)',   f"{teacher_gp_rank1:.4f}",                   f"{student_gp_rank1:.4f}"],
]
print(tabulate(compare_rows, headers=['Metric', 'Teacher', 'Student (RKD v2)'], tablefmt='fancy_grid'))

## 11. Export ONNX

Export phần inference (backbone + embedding + L2 normalize).  
**ProjectionHead bị bỏ** — không tham gia inference.

In [ ]:
class InferenceWrapper(nn.Module):
    """Backbone + embedding + L2 normalize — không có MagLinear, không có projector."""
    def __init__(self, model):
        super().__init__()
        self.backbone  = model.backbone
        self.embedding = model.embedding

    def forward(self, x):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb, p=2, dim=1)


inference_model = InferenceWrapper(student).eval().cpu()
dummy_input = torch.randn(1, 3, 112, 112)

onnx_path = os.path.join(manager.ckpt_dir, 'rkd_v2_mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')